In [4]:
import pandas as pd
import sqlite3
from pathlib import Path

# Check what data files are available
print("📁 Available data files:")
data_dir = Path('data')
for file in data_dir.glob('*'):
    print(f"  - {file.name} ({file.stat().st_size} bytes)")

# Check raw and processed directories
for subdir in ['raw', 'processed']:
    subpath = data_dir / subdir
    if subpath.exists():
        print(f"\n📁 data/{subdir}/:")
        for file in subpath.glob('*'):
            print(f"  - {file.name} ({file.stat().st_size} bytes)")

📁 Available data files:
  - mutual_funds.db (0 bytes)
  - processed (64 bytes)

📁 data/processed/:


In [ ]:
query = """
SELECT n.amfi_code, f.scheme_name, f.category, f.sub_category, 
       n.date, n.nav, n.daily_return
FROM fact_nav n
JOIN dim_fund f ON n.amfi_code = f.amfi_code
WHERE n.daily_return IS NOT NULL
ORDER BY n.amfi_code, n.date
"""
df = pd.read_sql(query, conn)

print(f"Loaded {len(df):,} daily return records")
print(f"Date range: {df['date'].min()} to {df['date'].max()}")
print(f"Number of funds: {df['amfi_code'].nunique()}")

# Validate daily returns distribution
print("\nDaily Returns Distribution:")
print(df['daily_return'].describe().to_string())

# Check for extreme outliers
outliers = df[abs(df['daily_return']) > 0.10]
print(f"\nExtreme returns (>10% or <-10%): {len(outliers)} rows")

In [ ]:
def calculate_cagr(fund_df, years):
    """Calculate CAGR given years (e.g., 1, 3, 5)"""
    end_date = fund_df['date'].max()
    start_date = end_date - pd.DateOffset(years=years)
    
    start_nav = fund_df[fund_df['date'] >= start_date]['nav'].iloc[0] if len(fund_df[fund_df['date'] >= start_date]) > 0 else np.nan
    end_nav = fund_df['nav'].iloc[-1]
    
    if pd.notna(start_nav) and start_nav > 0:
        return (end_nav / start_nav) ** (1/years) - 1
    return np.nan

cagr_results = []
for code in df['amfi_code'].unique():
    fund_data = df[df['amfi_code'] == code].sort_values('date')
    fund_name = fund_data['scheme_name'].iloc[0]
    
    cagr_1yr = calculate_cagr(fund_data, 1)
    cagr_3yr = calculate_cagr(fund_data, 3)
    cagr_5yr = calculate_cagr(fund_data, 5)
    
    cagr_results.append({
        'amfi_code': code,
        'scheme_name': fund_name,
        'cagr_1yr_pct': cagr_1yr * 100 if pd.notna(cagr_1yr) else np.nan,
        'cagr_3yr_pct': cagr_3yr * 100 if pd.notna(cagr_3yr) else np.nan,
        'cagr_5yr_pct': cagr_5yr * 100 if pd.notna(cagr_5yr) else np.nan
    })

cagr_df = pd.DataFrame(cagr_results)
print("CAGR Calculation Complete")
print(cagr_df.head(10).to_string())
cagr_df.to_csv(PROCESSED_PATH / "cagr_returns.csv", index=False)
print(f"Saved: {PROCESSED_PATH}/cagr_returns.csv")

In [ ]:
def calculate_sharpe(fund_returns):
    """Annualized Sharpe ratio"""
    if len(fund_returns) < 2 or fund_returns.std() == 0:
        return np.nan
    excess_return = fund_returns.mean() - RISK_FREE_RATE / TRADING_DAYS
    return (excess_return / fund_returns.std()) * np.sqrt(TRADING_DAYS)

sharpe_results = []
for code in df['amfi_code'].unique():
    fund_returns = df[df['amfi_code'] == code]['daily_return'].dropna()
    sharpe = calculate_sharpe(fund_returns)
    
    sharpe_results.append({
        'amfi_code': code,
        'scheme_name': df[df['amfi_code'] == code]['scheme_name'].iloc[0],
        'sharpe_ratio': sharpe,
        'avg_daily_return': fund_returns.mean() * 100,
        'volatility': fund_returns.std() * np.sqrt(TRADING_DAYS) * 100
    })

sharpe_df = pd.DataFrame(sharpe_results)
sharpe_df = sharpe_df.sort_values('sharpe_ratio', ascending=False)

print("Top 10 Funds by Sharpe Ratio:")
print(sharpe_df.head(10)[['scheme_name', 'sharpe_ratio']].to_string())
sharpe_df.to_csv(PROCESSED_PATH / "sharpe_ratio.csv", index=False)
print(f"Saved: {PROCESSED_PATH}/sharpe_ratio.csv")

In [ ]:
# Load Nifty 100 benchmark data
benchmark_df = pd.read_csv("data/raw/10_benchmark_indices.csv")
benchmark_df['date'] = pd.to_datetime(benchmark_df['date'])

nifty100 = benchmark_df[benchmark_df['index_name'] == 'NIFTY100'].copy()
nifty100 = nifty100.sort_values('date')
nifty100['daily_return'] = nifty100['close_value'].pct_change()

print(f"Benchmark data: {len(nifty100)} days")
print(f"NIFTY100 range: {nifty100['date'].min()} to {nifty100['date'].max()}")

In [ ]:
def calculate_alpha_beta(fund_returns, benchmark_returns):
    """OLS regression: fund_return = alpha + beta * benchmark_return"""
    # Align dates
    common_idx = fund_returns.index.intersection(benchmark_returns.index)
    if len(common_idx) < 30:
        return np.nan, np.nan
    
    fund_aligned = fund_returns.loc[common_idx]
    bench_aligned = benchmark_returns.loc[common_idx]
    
    # Linear regression
    slope, intercept, r_value, p_value, std_err = stats.linregress(bench_aligned, fund_aligned)
    
    alpha_annualized = intercept * TRADING_DAYS * 100  # Convert to percentage
    beta = slope
    
    return alpha_annualized, beta

# Create daily returns pivot
returns_pivot = df.pivot(index='date', columns='amfi_code', values='daily_return')
benchmark_pivot = nifty100.set_index('date')['daily_return']

alpha_beta_results = []
for code in returns_pivot.columns:
    fund_returns = returns_pivot[code].dropna()
    alpha, beta = calculate_alpha_beta(fund_returns, benchmark_pivot)
    
    alpha_beta_results.append({
        'amfi_code': code,
        'scheme_name': df[df['amfi_code'] == code]['scheme_name'].iloc[0],
        'alpha_annualized_pct': alpha,
        'beta': beta
    })

alpha_beta_df = pd.DataFrame(alpha_beta_results)
alpha_beta_df = alpha_beta_df.sort_values('alpha_annualized_pct', ascending=False)

print("Top 10 Funds by Alpha (outperformance):")
print(alpha_beta_df.head(10)[['scheme_name', 'alpha_annualized_pct', 'beta']].to_string())
alpha_beta_df.to_csv(PROCESSED_PATH / "alpha_beta.csv", index=False)
print(f"Saved: {PROCESSED_PATH}/alpha_beta.csv")

In [ ]:
def calculate_max_drawdown(fund_nav):
    """Calculate maximum drawdown and find worst period"""
    running_max = fund_nav.expanding().max()
    drawdown = (fund_nav - running_max) / running_max
    max_dd = drawdown.min()
    max_dd_date = drawdown.idxmin()
    
    # Find peak date before drawdown
    peak_date = fund_nav[:max_dd_date].expanding().max().idxmax()
    
    return max_dd, peak_date, max_dd_date

drawdown_results = []
for code in df['amfi_code'].unique():
    fund_nav = df[df['amfi_code'] == code].set_index('date')['nav'].sort_index()
    fund_name = df[df['amfi_code'] == code]['scheme_name'].iloc[0]
    
    max_dd, peak_date, trough_date = calculate_max_drawdown(fund_nav)
    
    drawdown_results.append({
        'amfi_code': code,
        'scheme_name': fund_name,
        'max_drawdown_pct': max_dd * 100,
        'drawdown_peak_date': peak_date,
        'drawdown_trough_date': trough_date,
        'recovery_days': (trough_date - peak_date).days if pd.notna(trough_date) else np.nan
    })

drawdown_df = pd.DataFrame(drawdown_results)
drawdown_df = drawdown_df.sort_values('max_drawdown_pct')  # Most negative first

print("Worst 5 Drawdowns:")
print(drawdown_df.head(5)[['scheme_name', 'max_drawdown_pct', 'drawdown_peak_date']].to_string())
drawdown_df.to_csv(PROCESSED_PATH / "max_drawdown.csv", index=False)
print(f"Saved: {PROCESSED_PATH}/max_drawdown.csv")

In [ ]:
expense_df = pd.read_csv("data/raw/01_fund_master.csv")
expense_df = expense_df[['amfi_code', 'scheme_name', 'expense_ratio_pct']]

print("Expense ratios loaded:")
print(expense_df.head(5).to_string())

In [ ]:
# Merge all metrics
scorecard = sharpe_df[['amfi_code', 'scheme_name', 'sharpe_ratio']].copy()
scorecard = scorecard.merge(cagr_df[['amfi_code', 'cagr_3yr_pct']], on='amfi_code', how='left')
scorecard = scorecard.merge(alpha_beta_df[['amfi_code', 'alpha_annualized_pct', 'beta']], on='amfi_code', how='left')
scorecard = scorecard.merge(drawdown_df[['amfi_code', 'max_drawdown_pct']], on='amfi_code', how='left')
scorecard = scorecard.merge(expense_df[['amfi_code', 'expense_ratio_pct']], on='amfi_code', how='left')

# Calculate ranks (higher is better for all except expense and drawdown)
scorecard['rank_3yr_return'] = scorecard['cagr_3yr_pct'].rank(ascending=False)
scorecard['rank_sharpe'] = scorecard['sharpe_ratio'].rank(ascending=False)
scorecard['rank_alpha'] = scorecard['alpha_annualized_pct'].rank(ascending=False)
scorecard['rank_expense'] = scorecard['expense_ratio_pct'].rank(ascending=True)  # Lower expense = better
scorecard['rank_drawdown'] = scorecard['max_drawdown_pct'].rank(ascending=True)  # Less negative = better

# Normalize ranks to 0-100 scale
max_rank = len(scorecard)
for col in ['rank_3yr_return', 'rank_sharpe', 'rank_alpha', 'rank_expense', 'rank_drawdown']:
    scorecard[f'score_{col}'] = (1 - (scorecard[col] - 1) / (max_rank - 1)) * 100

# Calculate composite score (weights)
scorecard['composite_score'] = (
    0.30 * scorecard['score_rank_3yr_return'] +
    0.25 * scorecard['score_rank_sharpe'] +
    0.20 * scorecard['score_rank_alpha'] +
    0.15 * scorecard['score_rank_expense'] +
    0.10 * scorecard['score_rank_drawdown']
)

scorecard = scorecard.sort_values('composite_score', ascending=False)

print("Top 10 Funds by Composite Score (0-100):")
print(scorecard.head(10)[['scheme_name', 'composite_score', 'cagr_3yr_pct', 'sharpe_ratio', 'expense_ratio_pct']].to_string())
scorecard.to_csv(PROCESSED_PATH / "fund_scorecard.csv", index=False)
print(f"Saved: {PROCESSED_PATH}/fund_scorecard.csv")

In [ ]:
# Get top 5 funds by composite score
top_funds = scorecard.head(5)['amfi_code'].tolist()

# Get NAV data for top funds
nav_query = f"""
SELECT f.scheme_name, n.date, n.nav
FROM fact_nav n
JOIN dim_fund f ON n.amfi_code = f.amfi_code
WHERE n.amfi_code IN ({','.join(["'"+str(c)+"'" for c in top_funds])})
ORDER BY n.date
"""
top_nav = pd.read_sql(nav_query, conn)

# Normalize to 100 (show relative performance)
pivot_nav = top_nav.pivot(index='date', columns='scheme_name', values='nav')
normalized = pivot_nav / pivot_nav.iloc[0] * 100

# Get Nifty 100 normalized to 100
nifty_clean = nifty100[nifty100['date'] >= normalized.index.min()]
nifty_clean = nifty_clean.set_index('date')
nifty_normalized = nifty_clean['close_value'] / nifty_clean['close_value'].iloc[0] * 100

# Plot
plt.figure(figsize=(14, 7))
for col in normalized.columns:
    plt.plot(normalized.index, normalized[col], linewidth=1.5, label=col[:25])

plt.plot(nifty_normalized.index, nifty_normalized, linewidth=2.5, color='black', linestyle='--', label='NIFTY 100')

plt.xlabel('Date')
plt.ylabel('Normalized Performance (Base 100)')
plt.title('Top 5 Funds vs NIFTY 100 Benchmark (3-Year Performance)')
plt.legend(loc='upper left', fontsize=9)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(REPORTS_PATH / 'benchmark_comparison.png', dpi=150)
plt.close()

print("Saved: reports/benchmark_comparison.png")

# Calculate tracking error for top funds
print("\nTracking Error (annualized) vs NIFTY 100:")
for code in top_funds:
    fund_returns = returns_pivot[code].dropna()
    common_idx = fund_returns.index.intersection(benchmark_pivot.index)
    if len(common_idx) > 0:
        tracking_diff = fund_returns.loc[common_idx] - benchmark_pivot.loc[common_idx]
        tracking_error = tracking_diff.std() * np.sqrt(252) * 100
        fund_name = scorecard[scorecard['amfi_code'] == code]['scheme_name'].iloc[0]
        print(f"  {fund_name[:35]}: {tracking_error:.2f}%")

In [ ]:
print("\n" + "=" * 60)
print("PERFORMANCE METRICS SUMMARY")
print("=" * 60)

print(f"\nBest Sharpe Ratio: {sharpe_df.iloc[0]['scheme_name']} ({sharpe_df.iloc[0]['sharpe_ratio']:.2f})")
print(f"Best Sortino Ratio: {sortino_df.iloc[0]['scheme_name']} ({sortino_df.iloc[0]['sortino_ratio']:.2f})")
print(f"Highest Alpha: {alpha_beta_df.iloc[0]['scheme_name']} ({alpha_beta_df.iloc[0]['alpha_annualized_pct']:.2f}%)")
print(f"Lowest Expense: {expense_df.loc[expense_df['expense_ratio_pct'].idxmin(), 'scheme_name']} ({expense_df['expense_ratio_pct'].min():.2f}%)")
print(f"Smallest Drawdown: {drawdown_df.iloc[-1]['scheme_name']} ({drawdown_df.iloc[-1]['max_drawdown_pct']:.1f}%)")
print(f"\nTop Overall Fund: {scorecard.iloc[0]['scheme_name']} (Score: {scorecard.iloc[0]['composite_score']:.1f}/100)")

print("\n" + "=" * 60)
print("DAY 4 COMPLETE")
print("Outputs created:")
print("  - data/processed/cagr_returns.csv")
print("  - data/processed/sharpe_ratio.csv")
print("  - data/processed/sortino_ratio.csv")
print("  - data/processed/alpha_beta.csv")
print("  - data/processed/max_drawdown.csv")
print("  - data/processed/fund_scorecard.csv")
print("  - reports/benchmark_comparison.png")
print("=" * 60)

conn.close()